# 05 — Permutation-based robustness check

Because the outcome distributions contain many tied values, omnibus tests for
the seven outcome-period combinations surviving Benjamini-Hochberg correction
(see `03_gambling_outcomes_analysis.ipynb`) were accompanied by permutation-based
Kruskal-Wallis tests (10,000 resamples, random seed 0), which do not rely on the
asymptotic chi-square approximation. Reproduces Table S1.

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import kruskal, permutation_test

ARM_MAP = {"AA": "A", "BB": "B", "CC": "C", "EE": "E", "FF": "F", "CONTROLGROUP": "Control"}

def load(path):
    d = pd.read_csv(path, low_memory=False)
    d["Arm"] = d["CONTROL_GROUP"].map(ARM_MAP).fillna(d["CONTROL_GROUP"])
    return d

def psum(df, pf, wks):
    c = [f"{pf}_POST_{w}" for w in wks if f"{pf}_POST_{w}" in df.columns]
    return df[c].sum(axis=1, min_count=1)

def kw_stat(*g):
    return kruskal(*g).statistic

In [ ]:
df1 = load("P10_final.csv")
df2 = load("P11_final.csv")

# The seven BH-surviving cells identified in 03_gambling_outcomes_analysis.ipynb
CELLS = [
    (df1, ["A", "C", "E", "F", "Control"], "N_WAGERS_DAILY_W", range(40, 52), "Exp1 N wagers Wk40-51"),
    (df2, ["A", "B", "C", "Control"], "SUM_WAGERS_DAILY_W", [1], "Exp2 Sum wagers Wk1"),
    (df2, ["A", "B", "C", "Control"], "SUM_TL_W", [1], "Exp2 TL Wk1"),
    (df2, ["A", "B", "C", "Control"], "NR_DAYS_W", [1], "Exp2 Gambling days Wk1"),
    (df2, ["A", "B", "C", "Control"], "NR_DAYS_W", range(2, 13), "Exp2 Gambling days Wk2-12"),
    (df2, ["A", "B", "C", "Control"], "NR_DAYS_W", range(40, 52), "Exp2 Gambling days Wk40-51"),
    (df2, ["A", "B", "C", "Control"], "SUM_TL_W", range(40, 52), "Exp2 TL Wk40-51"),
]

print("=== Asymptotic vs permutation Kruskal-Wallis (10,000 resamples, seed=0) ===")
for df, arms, pf, wks, lab in CELLS:
    y = psum(df, pf, wks)
    sub = pd.DataFrame({"arm": df["Arm"], "y": y}).dropna()
    g = [sub.loc[sub.arm == a, "y"].values for a in arms]
    H, pa = kruskal(*g)
    res = permutation_test(g, kw_stat, permutation_type="independent",
                           n_resamples=10000, alternative="greater", random_state=0)
    print(f"{lab:32s} H={H:6.2f}  asym_p={pa:.4f}  perm_p={res.pvalue:.4f}")